# CSC447 Lab 2: MPI and CUDA on Colab
Runtime > Change runtime type > GPU. Upload the project's `c/`, `cuda/` folders and the five files of `data/` to `MyDrive/lab2/` (same layout as the repo).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/lab2 && cp -r /content/drive/MyDrive/lab2 /content/lab2
%cd /content/lab2
!apt-get install -y -qq libopenmpi-dev openmpi-bin > /dev/null

In [ ]:
!lscpu | egrep 'Model name|^CPU\(s\)|Thread|Core|Socket'
!free -h | head -2
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2
!mpirun --version | head -1
!gcc --version | head -1
!cat /etc/os-release | head -2

In [ ]:
!gcc -O2 -Wall -Wextra c/sequential.c -o sequential
!mpicc -O2 -Wall -Wextra c/mpi_rooms.c -o mpi_rooms
!nvcc -O2 cuda/cuda_rooms.cu -o cuda_rooms
print('built')

## Correctness: small file and edge cases

In [ ]:
MPI = "mpirun --allow-run-as-root --oversubscribe --bind-to none -np"
!echo temperatures_small.csv | ./sequential | head -20
!{MPI} 4 ./mpi_rooms data/temperatures_small.csv | head -25
!./cuda_rooms data/temperatures_small.csv 128 | head -25

In [ ]:
import csv
exp = [r[:4] for r in csv.reader(open('data/expected_small.csv'))]
!{MPI} 4 ./mpi_rooms data/temperatures_small.csv > /dev/null
!./cuda_rooms data/temperatures_small.csv 128 > /dev/null
for f in ['results_mpi.csv','results_cuda.csv']:
    print(f, 'matches expected_small:', list(csv.reader(open(f))) == exp)

In [ ]:
!head -4 data/temperatures_small.csv > data/small3.csv
!echo '--- MPI 12 rooms, P=8' && {MPI} 8 ./mpi_rooms data/temperatures_small.csv | egrep 'rank|Normal|Total'
!echo '--- MPI 3 rooms, P=4' && {MPI} 4 ./mpi_rooms data/small3.csv | egrep 'rank|Normal|Total|^ *[0-9]+ '
!echo '--- CUDA 12 rooms, B=128: 1 block, threads 12-127 idle' && ./cuda_rooms data/temperatures_small.csv 128 | egrep 'GPU|Normal|Total'

## Benchmarks (medium and large)

In [ ]:
import subprocess, os, csv, filecmp
files = ['temperatures_medium.csv','temperatures_large.csv']
exp = {r['dataset']: r for r in csv.DictReader(open('data/expected_summary.csv'))}

def totals_ok(name, path):
    c = {'Normal':0,'Watch':0,'Urgent':0}; hot = n = 0
    for r in csv.DictReader(open(path)):
        c[r['category']] += 1; hot += int(r['hot_readings']); n += 1
    e = exp[name]
    return (n, c['Normal'], c['Watch'], c['Urgent'], hot) == tuple(int(e[k]) for k in
        ('rooms','normal_rooms','watch_rooms','urgent_rooms','total_hot_readings'))

def run(cmd, stdin=None):
    out = subprocess.run(cmd, shell=True, input=stdin, capture_output=True, text=True).stdout
    line = [l for l in out.splitlines() if l.startswith('RESULT,')][0].split(',')
    kt = [l for l in out.splitlines() if l.startswith('Kernel-only')]
    return [float(x) for x in line[4:9]], float(line[9]), (kt[0] if kt else '')

rows = []
for f in files:
    d = 'data/' + f
    t, m, _ = run('./sequential', d + '\n')
    os.replace('results_sequential.csv', 'ref.csv')
    ok = totals_ok(f, 'ref.csv'); sm = m
    rows.append([f, 'sequential', 1] + t + [m, 1.0, ok, ''])
    print(f, 'sequential', f'{m:.6f}', ok)
    jobs = [('mpi', p, f'{MPI} {p} ./mpi_rooms {d}', 'results_mpi.csv') for p in (1,2,4,8)] + \
           [('cuda', b, f'./cuda_rooms {d} {b}', 'results_cuda.csv') for b in (128,256,512)]
    for ver, x, cmd, res in jobs:
        t, m, k = run(cmd)
        ok = filecmp.cmp(res, 'ref.csv', shallow=False) and totals_ok(f, res)
        rows.append([f, ver, x] + t + [m, sm/m, ok, k])
        print(f'  {ver} {x:4d} median {m:.6f} speedup {sm/m:.3f} correct {ok}  {k}')

with open('benchmark_results.csv','w',newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['dataset','version','P_or_B','t1','t2','t3','t4','t5','median','speedup','correct','kernel_only'])
    w.writerows(rows)

## Figures

In [ ]:
import matplotlib.pyplot as plt
for ver, xl, name in (('mpi','Number of MPI processes P','speedup_mpi'), ('cuda','Threads per block B','speedup_cuda')):
    fig, ax = plt.subplots(figsize=(5.6,3.7), dpi=200)
    for f, c in zip(files, ('#1f5fa8','#c4561a')):
        pts = [(r[2], r[-3]) for r in rows if r[0]==f and r[1]==ver]
        if ver == 'mpi': pts = [(1,1.0)] + pts
        ax.plot(*zip(*pts), marker='o', color=c, label=f.replace('temperatures_','').replace('.csv',''))
    ax.axhline(1, color='gray', ls=':', lw=0.8, label='sequential = 1')
    ax.set_xlabel(xl); ax.set_ylabel('Speedup vs sequential'); ax.set_xscale('log', base=2)
    ax.set_xticks([r[2] for r in rows if r[1]==ver][:4 if ver=='mpi' else 3]); ax.set_xticklabels(ax.get_xticks())
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(name + '.png'); fig.savefig(name + '.pdf')
plt.show()
!mkdir -p /content/drive/MyDrive/lab2/out && cp benchmark_results.csv speedup_* /content/drive/MyDrive/lab2/out/